# Generative Adversarial Networks — Lab Notebook

This lab builds a GAN from scratch on **CIFAR-10** (32×32 RGB images, 10 object
classes) and explores the key properties that distinguish GANs from other
generative models.

**What you will implement and explore:**
1. Vanilla GAN — Generator vs Discriminator, adversarial training loop
2. Training dynamics — loss curves, mode collapse, discriminator saturation
3. Generated samples — evolution across training epochs
4. Latent space interpolation — smooth walks through the generator
5. DCGAN — convolutional architecture for sharper RGB images
6. Conditional GAN (cGAN) — controlling which object class is generated
7. GAN vs VAE comparison — sharpness vs diversity

**Runtime:** use GPU (Runtime → Change runtime type → T4 GPU). Training takes ~5 min per model.

---
### Why CIFAR-10?
- 32×32 **colour** images → forces a convolutional (DCGAN) architecture
- 10 **object classes** (planes, cars, birds, cats, deer, dogs, frogs, horses, ships, trucks)
  → makes the conditional GAN extension immediately meaningful
- Generating convincing objects is hard enough to expose **mode collapse** and
  training instability clearly — unlike MNIST digits which are too easy

---
### Key GAN idea (recap)
A GAN trains two networks simultaneously:
- **Generator G**: maps random noise z ~ N(0,I) → fake image. Wants to fool D.
- **Discriminator D**: classifies real vs fake. Wants to catch G.

They play a minimax game until G produces images D cannot distinguish from real ones.


## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
LATENT_DIM = 100      # dimensionality of the noise vector z
IMG_CH     = 3        # CIFAR-10 is RGB
IMG_SIZE   = 32       # spatial size
BATCH_SIZE = 128
EPOCHS     = 50

# CIFAR-10 class names for labelling plots
CLASSES = ['plane','car','bird','cat','deer',
           'dog','frog','horse','ship','truck']

torch.manual_seed(42)
np.random.seed(42)
print(f'Device: {DEVICE}')


## 1. Data — CIFAR-10

CIFAR-10 contains 60,000 32×32 RGB images across 10 classes (50k train / 10k test).

**Normalisation:** we map pixel values to **[-1, 1]** using `Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))`.
This matches the **Tanh** output of the generator, which also lives in [-1,1].
Using the same range for real and fake images is critical — if they live in different
ranges the discriminator has an unfair advantage from the start.


In [ ]:
# Normalise to [-1, 1] — matches Tanh output of the generator
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),   # mean per channel
                         (0.5, 0.5, 0.5))   # std  per channel  →  maps [0,1] → [-1,1]
])

train_data   = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

print(f'Training samples : {len(train_data)}')
print(f'Image shape      : {train_data[0][0].shape}  (C x H x W)')
print(f'Batches per epoch: {len(train_loader)}')

# Show a grid of real images
real_batch, real_labels = next(iter(train_loader))
grid = make_grid(real_batch[:32], nrow=8, normalize=True, value_range=(-1,1))
fig, ax = plt.subplots(figsize=(10,4))
ax.imshow(grid.permute(1,2,0))
ax.set_title('Real CIFAR-10 images  (' +
             '  '.join(CLASSES[l] for l in real_labels[:8].tolist()) + '  ...)',
             fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()


## 2. DCGAN Architecture

CIFAR-10 is 32×32 RGB, so we use **DCGAN** directly — plain MLP GANs cannot
capture spatial structure in colour images.

### 2.1 Generator
Maps noise **z** ∈ ℝ¹⁰⁰ → fake image ∈ ℝ³ˣ³²ˣ³².
Uses `ConvTranspose2d` (transposed convolution) to progressively upsample:
`4×4 → 8×8 → 16×16 → 32×32`

Key design choices:
- **ReLU** in hidden layers (standard for generators)
- **Tanh** output — maps to [-1,1], matching the normalised real images
- **BatchNorm2d** after each hidden layer (stabilises gradient flow)
- **No BatchNorm** on the output layer


In [ ]:
class Generator(nn.Module):
    """DCGAN Generator: z (B,100,1,1) → image (B,3,32,32)"""
    def __init__(self, latent_dim=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            # z: (B,100,1,1) → (B, ngf*8, 4, 4)
            nn.ConvTranspose2d(latent_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),

            # (B, ngf*8, 4, 4) → (B, ngf*4, 8, 8)   stride=2 doubles spatial size
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),

            # (B, ngf*4, 8, 8) → (B, ngf*2, 16, 16)
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),

            # (B, ngf*2, 16, 16) → (B, 3, 32, 32)
            nn.ConvTranspose2d(ngf*2, 3, 4, 2, 1, bias=False),
            nn.Tanh()   # output in [-1, 1]
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


# Sanity check
G_test = Generator()
z_test = torch.randn(4, 100)
out    = G_test(z_test)
print(f'Generator output shape: {out.shape}')           # (4, 3, 32, 32)
print(f'Output range: [{out.min():.2f}, {out.max():.2f}]')  # near (-1, 1)


### 2.2 Discriminator
Maps image ∈ ℝ³ˣ³²ˣ³² → scalar probability of being real.
Uses **strided convolutions** to downsample (no pooling — recommended for GANs):
`32×32 → 16×16 → 8×8 → 4×4 → 1×1`

Key design choices:
- **LeakyReLU(0.2)** — allows small gradients for negative activations, preventing dead neurons
- **No BatchNorm** on the first layer (standard DCGAN guideline)
- **Sigmoid** output — probability ∈ [0,1]


In [ ]:
class Discriminator(nn.Module):
    """DCGAN Discriminator: image (B,3,32,32) → probability (B,1)"""
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            # (B,3,32,32) → (B, ndf, 16, 16)   no BatchNorm on first layer
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (B, ndf, 16, 16) → (B, ndf*2, 8, 8)
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),

            # (B, ndf*2, 8, 8) → (B, ndf*4, 4, 4)
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),

            # (B, ndf*4, 4, 4) → (B, 1, 1, 1)
            nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img).view(-1, 1)


# Weight initialisation: N(0, 0.02) for conv, 1/0 for batchnorm
def weights_init(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


D_test   = Discriminator()
img_test = torch.randn(4, 3, 32, 32)
score    = D_test(img_test)
print(f'Discriminator output shape: {score.shape}')  # (4, 1)
print(f'Score range: [{score.min():.3f}, {score.max():.3f}]')


## 3. GAN Training Loop

The GAN trains two networks with **two separate optimisers** and **two separate loss steps** per batch:

**Step A — Train Discriminator:**
- Feed real images → D should output 1 (real)
- Feed fake images from G → D should output 0 (fake)
- Loss: `BCE(D(real), 1) + BCE(D(G(z)), 0)`
- Update **only D weights**

**Step B — Train Generator:**
- Feed new fake images → D should output 1 (generator wants to fool D)
- Loss: `BCE(D(G(z)), 1)`  — generator maximises D's mistake
- Update **only G weights**

**Critical:** G and D must be updated independently. Do not propagate G's gradient through D's update and vice versa.

In [ ]:
def train_gan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM):
    G = Generator(latent_dim).to(DEVICE);  G.apply(weights_init)
    D = Discriminator().to(DEVICE);        D.apply(weights_init)

    # Separate optimisers — each only updates its own network
    # beta1=0.5 is the standard DCGAN recommendation (vs default 0.9)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

    criterion = nn.BCELoss()

    # Fixed noise for tracking progress (same z every epoch)
    fixed_z = torch.randn(64, latent_dim).to(DEVICE)

    history   = {'d_loss': [], 'g_loss': [], 'd_real': [], 'd_fake': []}
    snapshots = {}

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_d = ep_g = ep_dr = ep_df = 0

        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)

            real_labels = torch.ones(B, 1).to(DEVICE)
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            # ── Step A: Train Discriminator ──────────────────────────────
            opt_D.zero_grad()

            d_real    = D(real_imgs)
            loss_real = criterion(d_real, real_labels)

            z         = torch.randn(B, latent_dim).to(DEVICE)
            fake_imgs = G(z).detach()   # .detach() stops gradients flowing into G
            d_fake    = D(fake_imgs)
            loss_fake = criterion(d_fake, fake_labels)

            loss_D = loss_real + loss_fake
            loss_D.backward()
            opt_D.step()

            # ── Step B: Train Generator ──────────────────────────────────
            opt_G.zero_grad()

            z         = torch.randn(B, latent_dim).to(DEVICE)
            fake_imgs = G(z)
            g_out     = D(fake_imgs)
            loss_G    = criterion(g_out, real_labels)   # fool the discriminator

            loss_G.backward()
            opt_G.step()

            ep_d  += loss_D.item();  ep_g  += loss_G.item()
            ep_dr += d_real.mean().item(); ep_df += d_fake.mean().item()

        n = len(train_loader)
        history['d_loss'].append(ep_d / n)
        history['g_loss'].append(ep_g / n)
        history['d_real'].append(ep_dr / n)
        history['d_fake'].append(ep_df / n)

        if epoch in [1, 5, 10, 20, 30, 50] or epoch == epochs:
            G.eval()
            with torch.no_grad():
                snapshots[epoch] = G(fixed_z).cpu()
            G.train()

        print(f'Epoch {epoch:3d}/{epochs} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f} '
              f'| D(real)={ep_dr/n:.3f}  D(fake)={ep_df/n:.3f}')

    return G, D, history, snapshots


print('Training DCGAN on CIFAR-10 ...')
G, D, history, snapshots = train_gan()


## 4. Training Dynamics

### Understanding the loss curves

- **D loss** ≈ log(2) ≈ 0.693 at equilibrium — D is guessing at chance level
- **G loss** high early (D easily catches fakes), should decrease as G improves
- **D(real)** = average score D assigns to real images (should stay near 0.5–0.7)
- **D(fake)** = average score D assigns to fake images (should rise as G improves)

**Warning signs:**
- D loss → 0: discriminator has won — G cannot fool it. Often means mode collapse.
- G loss → 0: unlikely, means G perfectly fools D (or D is broken).
- D(real) → 1 and D(fake) → 0 simultaneously: D has completely dominated G.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, len(history['d_loss']) + 1)

# Losses
axes[0].plot(epochs_range, history['d_loss'], label='D loss', color='crimson')
axes[0].plot(epochs_range, history['g_loss'], label='G loss', color='steelblue')
axes[0].axhline(0.693, color='gray', linestyle='--', lw=0.8, label='log(2) equilibrium')
axes[0].set_title('GAN losses'); axes[0].set_xlabel('Epoch')
axes[0].legend()

# D(real) and D(fake) — measure of balance
axes[1].plot(epochs_range, history['d_real'], label='D(real)', color='seagreen')
axes[1].plot(epochs_range, history['d_fake'], label='D(fake)', color='darkorange')
axes[1].axhline(0.5, color='gray', linestyle='--', lw=0.8, label='0.5 equilibrium')
axes[1].set_title('Discriminator scores'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend()

plt.tight_layout()
plt.savefig('gan_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Generated Images — Evolution Across Training

The **same fixed noise vector** z is decoded at each snapshot epoch.  
This lets us see how the same "latent identity" improves across training.

In [ ]:
def show_snapshots(snapshots, n_col=8):
    epochs_shown = sorted(snapshots.keys())
    fig, axes = plt.subplots(len(epochs_shown), 1,
                             figsize=(n_col * 1.2, len(epochs_shown) * 1.4))
    for ax, ep in zip(axes, epochs_shown):
        imgs = snapshots[ep][:n_col]           # first n_col images
        grid = make_grid(imgs, nrow=n_col,
                         normalize=True,       # maps [-1,1] → [0,1] for display
                         value_range=(-1, 1))
        ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
        ax.set_ylabel(f'Epoch {ep}', fontsize=11)
        ax.axis('off')
    plt.suptitle('Generated samples (same z, different epochs)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('gan_evolution.png', dpi=120, bbox_inches='tight')
    plt.show()


show_snapshots(snapshots)

## 6. Final Generated Samples

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM).to(DEVICE)
    samples = G(z).cpu()

grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
ax.axis('off')
ax.set_title('GAN — 64 generated samples', fontsize=14)
plt.tight_layout()
plt.savefig('gan_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Latent Space Interpolation

Unlike VAEs, GANs have no explicit encoder.  
But the **generator** still defines a smooth mapping z → image.  
We can interpolate between two noise vectors and see the transition.

In [ ]:
def interpolate_gan(G, z1, z2, steps=10):
    """Linear interpolation between two noise vectors."""
    G.eval()
    alphas = torch.linspace(0, 1, steps)
    imgs   = []
    with torch.no_grad():
        for a in alphas:
            z = (1 - a) * z1 + a * z2
            imgs.append(G(z.unsqueeze(0).to(DEVICE)).cpu().squeeze())
    return imgs


# 3 independent interpolations
STEPS = 12
fig, axes = plt.subplots(3, STEPS, figsize=(STEPS * 1.2, 4.5))

for row in range(3):
    z1 = torch.randn(LATENT_DIM)
    z2 = torch.randn(LATENT_DIM)
    interp = interpolate_gan(G, z1, z2, steps=STEPS)
    for col, img in enumerate(interp):
        axes[row, col].imshow(
            img.permute(1,2,0).clamp(-1,1)
            * 0.5 + 0.5)  # [-1,1] → [0,1], RGB order
        axes[row, col].axis('off')

plt.suptitle('GAN latent interpolation (z₁ → z₂)', fontsize=13)
plt.tight_layout()
plt.savefig('gan_interpolation.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Larger DCGAN — More Capacity for CIFAR-10

The model above is already a DCGAN. Here we train a **wider** version (ngf=128)
to see whether extra capacity improves sample quality.

We also add **label smoothing** (real labels = 0.9 instead of 1.0) — a simple
regularisation trick that prevents the discriminator from becoming overconfident
and helps reduce mode collapse.


In [ ]:
def train_wider_gan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM, ngf=128):
    """Wider DCGAN with label smoothing.""\"
    G_w = Generator(latent_dim, ngf=ngf).to(DEVICE);  G_w.apply(weights_init)
    D_w = Discriminator(ndf=ngf).to(DEVICE);           D_w.apply(weights_init)

    opt_G = torch.optim.Adam(G_w.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D_w.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion  = nn.BCELoss()
    fixed_z    = torch.randn(64, latent_dim).to(DEVICE)
    snapshots_w = {}

    for epoch in range(1, epochs + 1):
        G_w.train(); D_w.train()
        ep_d = ep_g = 0
        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)
            # Label smoothing: real = 0.9 instead of 1.0
            real_labels = torch.full((B,1), 0.9).to(DEVICE)
            fake_labels = torch.zeros(B,1).to(DEVICE)

            opt_D.zero_grad()
            z = torch.randn(B, latent_dim).to(DEVICE)
            loss_D = (criterion(D_w(real_imgs), real_labels) +
                      criterion(D_w(G_w(z).detach()), fake_labels))
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad()
            z = torch.randn(B, latent_dim).to(DEVICE)
            loss_G = criterion(D_w(G_w(z)), torch.ones(B,1).to(DEVICE))
            loss_G.backward(); opt_G.step()

            ep_d += loss_D.item(); ep_g += loss_G.item()

        n = len(train_loader)
        if epoch in [1, 5, 10, 20, 30, 50] or epoch == epochs:
            G_w.eval()
            with torch.no_grad():
                snapshots_w[epoch] = G_w(fixed_z).cpu()
            G_w.train()
        print(f'Wider DCGAN Epoch {epoch:3d} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f}')

    return G_w, D_w, snapshots_w


print('Training wider DCGAN with label smoothing ...')
G_w, D_w, snapshots_w = train_wider_gan()


In [ ]:
# Compare baseline DCGAN vs wider DCGAN with label smoothing
G.eval(); G_w.eval()
with torch.no_grad():
    z = torch.randn(32, LATENT_DIM).to(DEVICE)
    base_samples  = G(z).cpu()
    wider_samples = G_w(z).cpu()

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
for ax, imgs, title in zip(axes,
                            [base_samples, wider_samples],
                            ['Baseline DCGAN (ngf=64)',
                             'Wider DCGAN + label smoothing (ngf=128)']):
    grid = make_grid(imgs, nrow=16, normalize=True, value_range=(-1,1))
    ax.imshow(grid.permute(1,2,0))
    ax.set_ylabel(title, fontsize=11); ax.axis('off')

plt.suptitle('Baseline vs Wider DCGAN — CIFAR-10', fontsize=13)
plt.tight_layout()
plt.savefig('dcgan_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Conditional GAN (cGAN)

A standard GAN generates objects randomly — we cannot control which class appears.
A **conditional GAN** conditions both G and D on the class label y:

- **Generator**: receives `concat(z, embed(y))` → generates object of class y
- **Discriminator**: receives `concat(img_features, embed(y))` → real/fake *given* class y

On CIFAR-10 this means we can request: `G(z, y='car')` → always generates a car.
The 10 visually distinct classes (planes, dogs, ships, ...) make the conditioning
much more compelling than MNIST digits.


In [ ]:
class CondGenerator(nn.Module):
    def __init__(self, latent_dim=100, n_classes=10, embed_dim=50, ngf=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, embed_dim)
        # After concat z + embed(y): input channels = latent_dim + embed_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim + embed_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        c   = self.label_emb(labels)               # (B, embed_dim)
        # Concatenate along channel dim for ConvTranspose input
        inp = torch.cat([z, c], dim=1)             # (B, latent_dim + embed_dim)
        return self.net(inp.view(inp.size(0), -1, 1, 1))


class CondDiscriminator(nn.Module):
    def __init__(self, n_classes=10, embed_dim=50, ndf=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, embed_dim)
        # Project embed to spatial map and concat as extra channel
        self.label_proj = nn.Linear(embed_dim, 32*32)
        self.net = nn.Sequential(
            # Input: 3+1=4 channels (image + label map)
            nn.Conv2d(4, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        c   = self.label_emb(labels)                      # (B, embed_dim)
        lmap = self.label_proj(c).view(-1, 1, 32, 32)    # (B, 1, 32, 32)
        inp  = torch.cat([img, lmap], dim=1)              # (B, 4, 32, 32)
        return self.net(inp).view(-1, 1)


def train_cgan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM):
    G_c = CondGenerator(latent_dim).to(DEVICE);  G_c.apply(weights_init)
    D_c = CondDiscriminator().to(DEVICE);         D_c.apply(weights_init)
    opt_G = torch.optim.Adam(G_c.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D_c.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    # Fixed: 5 samples per class, columns = classes
    fixed_z   = torch.randn(50, latent_dim).to(DEVICE)
    fixed_lbl = torch.arange(10).repeat(5).to(DEVICE)   # 0,1,...,9,0,1,...

    for epoch in range(1, epochs + 1):
        G_c.train(); D_c.train()
        ep_d = ep_g = 0
        for real_imgs, labels in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            labels    = labels.to(DEVICE)
            B = real_imgs.size(0)
            real_labels = torch.full((B,1), 0.9).to(DEVICE)  # label smoothing
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            opt_D.zero_grad()
            z      = torch.randn(B, latent_dim).to(DEVICE)
            fake_y = torch.randint(0, 10, (B,)).to(DEVICE)
            loss_D = (criterion(D_c(real_imgs, labels), real_labels) +
                      criterion(D_c(G_c(z, fake_y).detach(), fake_y), fake_labels))
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad()
            z      = torch.randn(B, latent_dim).to(DEVICE)
            fake_y = torch.randint(0, 10, (B,)).to(DEVICE)
            loss_G = criterion(D_c(G_c(z, fake_y), fake_y),
                               torch.ones(B,1).to(DEVICE))
            loss_G.backward(); opt_G.step()

            ep_d += loss_D.item(); ep_g += loss_G.item()

        n = len(train_loader)
        print(f'cGAN Epoch {epoch:3d} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f}')

    return G_c, D_c, fixed_z, fixed_lbl


print('Training conditional GAN on CIFAR-10 ...')
G_c, D_c, fixed_z_c, fixed_lbl_c = train_cgan()


In [ ]:
# Each column = one CIFAR-10 class, each row = same z with different class label
G_c.eval()
with torch.no_grad():
    cond_imgs = G_c(fixed_z_c, fixed_lbl_c).cpu()

grid = make_grid(cond_imgs, nrow=10, normalize=True, value_range=(-1,1))
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(grid.permute(1,2,0))
ax.set_title('cGAN — each column is one class (left to right: ' +
             ', '.join(CLASSES) + ')', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('cgan_cifar10.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Mode Collapse — What It Looks Like

**Mode collapse** is the most common GAN failure: the generator learns to produce
only a few (or one) type of output, ignoring the diversity of the training data.

We simulate it by training with a very high learning rate and no regularisation.

In [ ]:
def train_collapsed_gan(epochs=15, lr=5e-3):  # deliberately bad hyperparams
    G_bad = Generator(LATENT_DIM).to(DEVICE)
    D_bad = Discriminator().to(DEVICE)
    # High LR + default Adam betas (no DCGAN tuning) → collapse-prone
    opt_G = torch.optim.Adam(G_bad.parameters(), lr=lr)
    opt_D = torch.optim.Adam(D_bad.parameters(), lr=lr)
    criterion = nn.BCELoss()

    fixed_z = torch.randn(64, LATENT_DIM).to(DEVICE)
    snapshots_bad = {}

    for epoch in range(1, epochs + 1):
        G_bad.train(); D_bad.train()
        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)
            rl = torch.ones(B,1).to(DEVICE); fl = torch.zeros(B,1).to(DEVICE)
            opt_D.zero_grad()
            z = torch.randn(B, LATENT_DIM).to(DEVICE)
            (criterion(D_bad(real_imgs), rl) +
             criterion(D_bad(G_bad(z).detach()), fl)).backward()
            opt_D.step()
            opt_G.zero_grad()
            z = torch.randn(B, LATENT_DIM).to(DEVICE)
            criterion(D_bad(G_bad(z)), rl).backward()
            opt_G.step()

        if epoch in [1, 5, 10, 15]:
            G_bad.eval()
            with torch.no_grad():
                snapshots_bad[epoch] = G_bad(fixed_z).cpu()
            G_bad.train()
        print(f'Collapsed GAN epoch {epoch}')

    return snapshots_bad


print('Simulating mode collapse on CIFAR-10 ...')
snapshots_bad = train_collapsed_gan()

# Show the collapse evolution
show_snapshots(snapshots_bad)


## 11. GAN vs VAE — Visual Comparison on CIFAR-10

The GAN vs VAE trade-off is especially visible on colour images:
- **GAN**: sharp textures, vivid colours — but may produce artefacts or collapse to a few modes
- **VAE**: blurrier (pixel-level reconstruction loss averages over uncertainty), but stable and diverse

We train a quick convolutional VAE on CIFAR-10 to compare directly.


In [ ]:
# ── Quick convolutional VAE on CIFAR-10 for comparison ──────────────────────
class ConvVAEEncoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),    # 32→16
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),   # 16→8
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),  # 8→4
        )
        self.fc_mu  = nn.Linear(128*4*4, latent_dim)
        self.fc_lv  = nn.Linear(128*4*4, latent_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_lv(h)


class ConvVAEDecoder(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.fc  = nn.Linear(latent_dim, 128*4*4)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),   # 4→8
            nn.ConvTranspose2d(64,  32, 4, 2, 1), nn.ReLU(),   # 8→16
            nn.ConvTranspose2d(32,   3, 4, 2, 1), nn.Tanh(),   # 16→32
        )

    def forward(self, z):
        return self.net(self.fc(z).view(-1, 128, 4, 4))


class ConvVAE(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.enc = ConvVAEEncoder(latent_dim)
        self.dec = ConvVAEDecoder(latent_dim)

    def forward(self, x):
        mu, lv = self.enc(x)
        z = mu + torch.exp(0.5*lv) * torch.randn_like(mu)
        return self.dec(z), mu, lv


vae = ConvVAE(latent_dim=100).to(DEVICE)
opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)

print('Training convolutional VAE on CIFAR-10 for comparison ...')
for epoch in range(1, 21):
    for x, _ in train_loader:
        x = (x.to(DEVICE) + 1) / 2   # shift back to [0,1] for BCE
        opt_vae.zero_grad()
        xhat, mu, lv = vae(x*2 - 1)   # model expects [-1,1]
        xhat_01 = (xhat + 1) / 2
        recon = F.binary_cross_entropy(xhat_01.clamp(1e-6, 1-1e-6), x,
                                       reduction='sum') / x.size(0)
        kl    = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / x.size(0)
        (recon + kl).backward()
        opt_vae.step()
    print(f'  VAE epoch {epoch}')

# Compare GAN vs VAE on CIFAR-10
G.eval(); vae.eval()
with torch.no_grad():
    z_gan = torch.randn(32, LATENT_DIM).to(DEVICE)
    z_vae = torch.randn(32, 100).to(DEVICE)
    gan_imgs = G(z_gan).cpu()
    vae_imgs = vae.dec(z_vae).cpu()

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
for ax, imgs, title in zip(axes,
                            [gan_imgs, vae_imgs],
                            ['GAN  — implicit p(x),  sharp but unstable',
                             'VAE  — explicit p(x),  blurry but stable']):
    grid = make_grid(imgs, nrow=16, normalize=True, value_range=(-1,1))
    ax.imshow(grid.permute(1,2,0))
    ax.set_ylabel(title, fontsize=11); ax.axis('off')

plt.suptitle('GAN vs VAE on CIFAR-10 — sharpness vs diversity', fontsize=13)
plt.tight_layout()
plt.savefig('gan_vs_vae_cifar10.png', dpi=120, bbox_inches='tight')
plt.show()


## 12. Summary

| Property | Vanilla/DCGAN | Wider DCGAN + smoothing | cGAN | VAE |
|---|---|---|---|---|
| Dataset | CIFAR-10 (32×32 RGB) | CIFAR-10 | CIFAR-10 | CIFAR-10 |
| Explicit p(x)? | No (implicit) | No | No | Yes |
| Controllable class? | No | No | Yes (by label) | No |
| Sample sharpness | Medium–High | Higher | Medium | Low (blurry) |
| Training stability | Fragile | More stable | Fragile | Stable |
| Mode collapse risk | High | Lower | Medium | Very low |

**Key takeaways:**
1. **Adversarial training** produces sharper images than VAEs because there is no explicit pixel-level reconstruction loss — the discriminator acts as a learned perceptual judge.
2. **`.detach()`** on fake images in the D update is critical — without it, D gradients flow into G during D's update, corrupting training.
3. **beta1=0.5** in Adam (instead of default 0.9) is the standard DCGAN recommendation — it reduces momentum, helping with the non-stationary adversarial objective.
4. **D(real) ≈ D(fake) ≈ 0.5** at equilibrium. If D(real)→1 and D(fake)→0 simultaneously, the discriminator has dominated and mode collapse is likely.
5. **Label smoothing** (0.9 instead of 1.0 for real labels) is a simple regularisation that prevents the discriminator from being overconfident and helps training stability.
6. **Colour images** expose GAN failure modes much more clearly than MNIST — artefacts, colour bleed, and mode collapse are immediately visible.


In [ ]:
# ── Exercises ──────────────────────────────────────────────────────────────
# 1. Remove .detach() from fake_imgs in the D update and observe what happens
#    to the G loss. Why does it become unstable?
#
# 2. In the cGAN, fix z and vary the label from 0 to 9 (plane→car→...→truck).
#    Does the same z produce a consistent 'style' across different classes?
#
# 3. Add a spectral normalisation layer (nn.utils.spectral_norm) to the
#    discriminator. Does training become more stable?
#
# 4. Try training with n_D=2 (two D updates per G update) and n_D=5.
#    How does the D/G balance affect sample quality?
#
# 5. Replace BCELoss with the Wasserstein loss (WGAN):
#    loss_D = -D(real).mean() + D(fake).mean()
#    loss_G = -D(fake).mean()
#    Does it reduce mode collapse on CIFAR-10?
